# Module B Implementation and Optimization Report

> CS 432 Databases - Assignment 2 (Module B)

This report documents implementation evidence for SubTask 1 to SubTask 5:
- SubTask 1: Local environment setup and data integrity
- SubTask 2: Session-validated API and UI
- SubTask 3: RBAC and security logging
- SubTask 4: SQL indexing and query optimization
- SubTask 5: Before/after benchmarking and EXPLAIN analysis

## 1) Schema Design and Data Integrity (SubTask 1)

Core/project separation used in this module:
- Core identity/auth tables: `Member`, `AuthCredential`
- Core relation tables: `Follow`, `GroupMember`, `Group`
- Project content tables: `Post`, `Comment`

Integrity design choices:
- Credentials are not duplicated in project tables (stored only in `AuthCredential`).
- Foreign keys with cascade policies maintain consistency during create/delete operations.
- Constraint and trigger rules enforce data validity and business rules.

Member lifecycle requirement coverage:
- Admin member creation writes to both `Member` and `AuthCredential`.
- Member deletion cascades through related mappings/content as defined by schema constraints.

## 2) Security, Session Validation, and RBAC (SubTask 2 and 3)

Session validation:
- All protected endpoints validate a local JWT session token (`session-token` header).
- Invalid, missing, or expired sessions return `401`.

RBAC model:
- Admin users: member management, group membership administration, log inspection.
- Regular users: can modify only their own portfolio/posts/comments.
- Unauthorized cross-user modification attempts return `403`.

Security logging:
- API audit file: `Module_B/logs/audit.log`.
- DB-side write tracking: `ApiWriteLog` + triggers classify writes as `API` or `DIRECT_DB`.
- This makes direct SQL updates outside API/session validation easily identifiable as unauthorized.

## 3) SQL Indexing and Query Optimization (SubTask 4)

Frequently accessed / benchmarked endpoints:
- `GET /posts` (feed listing with filtering and ordering)
- `GET /posts/{post_id}/comments` (hotspot query under high-comment post)

Indexing strategy applied to API query clauses:
1. `idx_post_active_postdate_postid ON Post(IsActive, PostDate DESC, PostID DESC)`
   - Targets: `WHERE p.IsActive = TRUE` + `ORDER BY p.PostDate DESC, p.PostID DESC`
2. `idx_comment_post_active_date ON Comment(PostID, IsActive, CommentDate ASC)`
   - Targets: `WHERE c.PostID = ? AND c.IsActive = TRUE` + `ORDER BY c.CommentDate ASC`

Both are project-table indexes aligned to actual WHERE/JOIN/ORDER BY patterns.

## 4) Performance Benchmarking and EXPLAIN Evidence (SubTask 5)

Methodology:
- Captured SQL execution times and API response times for the same endpoints.
- Executed two stages with identical benchmark parameters:
  - `before_indexes`
  - `after_indexes`
- Captured EXPLAIN plans in both stages and compared access characteristics (`type`, `key`, `rows`, `extra`).

Reproducibility steps:
1. Start API server:
   - `cd Module_B/app`
   - `uvicorn main:app --reload --port 8001`
2. Run benchmark script:
   - `cd Module_B/app`
   - `python run_index_benchmark.py`
3. Inspect artifact:
   - `Module_B/performance/index_benchmark_results.json`

Interpretation note:
- Speedups may vary run-to-run depending on dataset size/distribution and cache effects.
- EXPLAIN output is the primary evidence of access-plan changes.

In [10]:
import json
from pathlib import Path

candidate_paths = [
    Path("performance/index_benchmark_results.json"),
    Path("index_benchmark_results.json"),
    Path("Module_B/performance/index_benchmark_results.json"),
]

results_path = next((p for p in candidate_paths if p.exists()), None)
if results_path is None:
    raise FileNotFoundError(
        "Could not find index_benchmark_results.json. Run Module_B/app/run_index_benchmark.py first."
    )

with results_path.open("r", encoding="utf-8") as f:
    results = json.load(f)

print(f"Loaded benchmark file: {results_path}")
results["benchmark_params"]

Loaded benchmark file: performance\index_benchmark_results.json


{'iterations': 40,
 'warmup': 5,
 'limit': 20,
 'offset': 1510,
 'comment_post_id': 3021}

In [15]:
from IPython.display import Markdown, display

before = results["stages"][0]
after = results["stages"][1]
speedup = results["speedup"]

def fmt(value):
    return "-" if value is None else f"{value:.3f}"

rows = [
    {
        "Endpoint": "GET /posts",
        "Layer": "SQL",
        "Before Avg (ms)": before["sql_ms"]["list_posts"]["avg_ms"],
        "After Avg (ms)": after["sql_ms"]["list_posts"]["avg_ms"],
        "Before Median (ms)": before["sql_ms"]["list_posts"].get("median_ms"),
        "After Median (ms)": after["sql_ms"]["list_posts"].get("median_ms"),
        "Speedup (x)": speedup["posts_sql_speedup"],
    },
    {
        "Endpoint": "GET /posts/{post_id}/comments",
        "Layer": "SQL",
        "Before Avg (ms)": before["sql_ms"]["list_comments"]["avg_ms"],
        "After Avg (ms)": after["sql_ms"]["list_comments"]["avg_ms"],
        "Before Median (ms)": before["sql_ms"]["list_comments"].get("median_ms"),
        "After Median (ms)": after["sql_ms"]["list_comments"].get("median_ms"),
        "Speedup (x)": speedup["comments_sql_speedup"],
    },
    {
        "Endpoint": "GET /posts",
        "Layer": "API",
        "Before Avg (ms)": before["api_ms"]["list_posts"]["avg_ms"],
        "After Avg (ms)": after["api_ms"]["list_posts"]["avg_ms"],
        "Before Median (ms)": before["api_ms"]["list_posts"].get("median_ms"),
        "After Median (ms)": after["api_ms"]["list_posts"].get("median_ms"),
        "Speedup (x)": speedup["posts_api_speedup"],
    },
    {
        "Endpoint": "GET /posts/{post_id}/comments",
        "Layer": "API",
        "Before Avg (ms)": before["api_ms"]["list_comments"]["avg_ms"],
        "After Avg (ms)": after["api_ms"]["list_comments"]["avg_ms"],
        "Before Median (ms)": before["api_ms"]["list_comments"].get("median_ms"),
        "After Median (ms)": after["api_ms"]["list_comments"].get("median_ms"),
        "Speedup (x)": speedup["comments_api_speedup"],
    },
]

headers = [
    "Endpoint",
    "Layer",
    "Before Avg (ms)",
    "After Avg (ms)",
    "Before Median (ms)",
    "After Median (ms)",
    "Speedup (x)",
]

table = [
    "| " + " | ".join(headers) + " |",
    "|" + "|".join(["---"] * len(headers)) + "|",
]

for row in rows:
    table.append(
        "| " + " | ".join(
            [
                str(row["Endpoint"]),
                str(row["Layer"]),
                fmt(row["Before Avg (ms)"]),
                fmt(row["After Avg (ms)"]),
                fmt(row["Before Median (ms)"]),
                fmt(row["After Median (ms)"]),
                fmt(row["Speedup (x)"]),
            ]
        ) + " |"
    )

display(Markdown("\n".join(table)))

| Endpoint | Layer | Before Avg (ms) | After Avg (ms) | Before Median (ms) | After Median (ms) | Speedup (x) |
|---|---|---|---|---|---|---|
| GET /posts | SQL | 6.158 | 5.452 | 4.938 | 4.148 | 1.129 |
| GET /posts/{post_id}/comments | SQL | 22.729 | 19.562 | 22.537 | 19.204 | 1.162 |
| GET /posts | API | 37.984 | 38.659 | 37.862 | 41.145 | 0.983 |
| GET /posts/{post_id}/comments | API | 92.428 | 88.416 | 94.252 | 89.307 | 1.045 |

In [16]:
from IPython.display import Markdown, display

def explain_rows(query_name, stage_name, data_rows):
    rows = []
    for r in data_rows:
        rows.append(
            {
                "Query": query_name,
                "Stage": stage_name,
                "Table": r.get("table"),
                "Type": r.get("type"),
                "Key": r.get("key"),
                "Rows": r.get("rows"),
                "Extra": r.get("extra"),
            }
        )
    return rows

rows = []
rows.extend(explain_rows("GET /posts", "before", before["explain"]["list_posts"]))
rows.extend(explain_rows("GET /posts", "after", after["explain"]["list_posts"]))
rows.extend(explain_rows("GET /posts/{post_id}/comments", "before", before["explain"]["list_comments"]))
rows.extend(explain_rows("GET /posts/{post_id}/comments", "after", after["explain"]["list_comments"]))

headers = ["Query", "Stage", "Table", "Type", "Key", "Rows", "Extra"]
table = [
    "| " + " | ".join(headers) + " |",
    "|" + "|".join(["---"] * len(headers)) + "|",
]

for row in rows:
    table.append(
        "| " + " | ".join(
            [
                str(row["Query"]),
                str(row["Stage"]),
                str(row["Table"]),
                str(row["Type"]),
                str(row["Key"]),
                str(row["Rows"]),
                str(row["Extra"]),
            ]
        ) + " |"
    )

display(Markdown("\n".join(table)))

| Query | Stage | Table | Type | Key | Rows | Extra |
|---|---|---|---|---|---|---|
| GET /posts | before | p | ALL | None | 3021 | Using where; Using filesort |
| GET /posts | before | m | eq_ref | PRIMARY | 1 | None |
| GET /posts | after | p | ALL | None | 3021 | Using where; Using filesort |
| GET /posts | after | m | eq_ref | PRIMARY | 1 | None |
| GET /posts/{post_id}/comments | before | c | ref | idx_comment_post | 2500 | Using where; Using filesort |
| GET /posts/{post_id}/comments | before | m | eq_ref | PRIMARY | 1 | None |
| GET /posts/{post_id}/comments | after | c | range | idx_comment_post_active_date | 2500 | Using index condition |
| GET /posts/{post_id}/comments | after | m | eq_ref | PRIMARY | 1 | None |

## 5) Video Demonstration Link

Add your hosted 3-5 minute demo link here (YouTube unlisted or open Drive link):
- `<PASTE_VIDEO_LINK_HERE>`

## 6) Conclusion

- Module B integrates secure local APIs, RBAC enforcement, and auditable DB operations.
- Indexing/benchmarking pipeline is implemented with before-vs-after quantitative evidence.
- Remaining result analysis can be expanded with larger datasets if needed.